# Piloto 2026 — validación final anidada del modelo de aromas

## tl;dr

**Veredicto:** `NO_VALID_MODEL`.  
**Modelo seleccionado:** `None`.

La comparación final contiene: recuperación constante, recuperación dependiente de etanol y esta última con un reservorio de línea que conserva masa. La calibración se pondera de acuerdo con el NRMSE del gate y se valida dejando fuera un reactor completo.

## Contrato de interpretación

- Producción: dos rendimientos ligados al consumo de azúcar; no se agrega aquí otro estado de pulso porque no fue transferible en la prueba anterior.
- Pérdida: partición Morakul/Mouret dependiente de etanol y temperatura, forzada por el rCO₂ validado.
- Observación: fracción efectiva de recuperación dependiente de etanol.
- Memoria: inventario de línea de primer orden; su τ sólo es físico si coincide con una medición de residencia.
- Validación: interna retrospectiva; una campaña prospectiva sellada sigue siendo obligatoria.

In [ ]:
from pathlib import Path
import os
import sys
from IPython.display import display, Image

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'fermentation_model').exists():
    ROOT = ROOT.parent
if not (ROOT / 'fermentation_model').exists():
    raise RuntimeError('Execute from the repository or a descendant directory')
sys.path.insert(0, str(ROOT / 'fermentation_model'))
from pilot_2026 import run_aroma_final_nested_validation_2026 as analysis
result = analysis.load_results() if os.environ.get('PILOT_AROMA_REUSE_RESULTS') == '1' else analysis.run_analysis()
print('Veredicto:', result['gate']['verdict'])
print('Modelo seleccionado:', result['gate']['selected_model'])
print('Mejor diagnóstico:', result['gate']['diagnostic_best_model'])

## Calidad y límite de reproducibilidad

In [ ]:
display(result['repeatability'].round(4))
display(result['anomaly_summary'].head(10).round(3))
display(Image(filename=analysis.FIGURE_DIR / '01_sample_anomaly_audit.png'))

## Validación cruzada y estabilidad

In [ ]:
display(result['metrics'].round(4))
display(result['comparison'].round(4))
display(result['stability'].round(5))
display(Image(filename=analysis.FIGURE_DIR / '02_capture_efficiency_loro_nrmse.png'))
display(Image(filename=analysis.FIGURE_DIR / '07_capture_efficiency_by_fold.png'))

## Forzantes y curvas finales

In [ ]:
display(Image(filename=analysis.FIGURE_DIR / '03_rco2_temperature_pulse_drivers.png'))
for species in analysis.decision.ethanol.capture.base.SPECIES_LABELS:
    display(Image(filename=analysis.FIGURE_DIR / f'04_liquid_{species}.png'))
    display(Image(filename=analysis.FIGURE_DIR / f'05_condensate_{species}.png'))

## Parámetros finales

In [ ]:
display(result['parameters'].round(6))
display(result['fit_validation'].round(6))

## Separación producción–transferencia–captura

In [ ]:
selected = result['gate']['selected_model'] or result['gate']['diagnostic_best_model']
display(result['mass_balance'].query('model_variant == @selected').round(4))

## Takeaways

- `PASS` indica transferibilidad interna bajo todos los guardrails; no convierte η o τ en parámetros físicos medidos.
- `NO_VALID_MODEL` indica que la variabilidad de condensado y/o la inestabilidad paramétrica impiden una conclusión predictiva con estos datos.
- La confirmación mínima mide simultáneamente vino, gas de salida, volumen y %EtOH de cada MIX, recuperación de estándar gaseoso y duplicados.

In [ ]:
assert result['gate']['verdict'] in {'PASS', 'NO_VALID_MODEL'}
assert len(result['figures']) == 8
assert result['fit_validation']['maximum_relative_mass_balance_error'].max() <= 1e-8
print('Notebook ejecutado sin errores; figuras:', len(result['figures']))